# Hemo Invasion PDE Demo

This tutorial shows how to run `HemoInvasion3D` in TumorTwin using the same pattern as `HGG_Demo` and `TNBC_Demo`.

We include a **mini 50-day run** for quick sanity checking.

In [ ]:
from datetime import timedelta
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from tumortwin.models import HemoInvasion3D
from tumortwin.optimizers import LMoptimizer, LMoptions
from tumortwin.preprocessing import ADC_to_cellularity
from tumortwin.solvers import TorchDiffEqSolver, TorchDiffEqSolverOptions
from tumortwin.types import CropSettings, CropTarget
from tumortwin.types.hgg_data import HGGPatientData

In [ ]:
# Choose device
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

# Resolve paths robustly (works whether cwd is repo root or tutorials/)
cwd = Path.cwd()
repo_root = cwd if (cwd / "tumortwin").exists() else cwd.parent

patient_json = repo_root / "input_files" / "HGG_demo_001" / "HGG_demo_001.json"
if not patient_json.exists():
    raise FileNotFoundError(f"Could not find patient json: {patient_json}")

crop_settings = CropSettings(crop_to=CropTarget.ROI_ENHANCE, padding=10, visit_index=-1)
patient_data = HGGPatientData.from_file(patient_json, crop_settings=crop_settings)

print(f"Patient: {patient_data.patient}")
print(f"Visits: {len(patient_data.visits)}")
print(f"Grid shape: {patient_data.brainmask_image.array.shape}")

In [ ]:
# Build initial fields from first visit
visit0 = patient_data.visits[0]

cellularity0 = ADC_to_cellularity(
    visit0.adc_image,
    visit0.roi_enhance_image,
    visit0.roi_nonenhance_image,
)

initial_n = torch.from_numpy(cellularity0.array).float().to(device)
initial_m = torch.zeros_like(initial_n)
initial_s = torch.ones_like(initial_n)

# We use ROI-enhancing voxels as a proxy vessel mask for the demo.
# In a production setup, replace this with a proper vessel segmentation/mask.
vessel_mask = torch.from_numpy((visit0.roi_enhance_image.array > 0).astype(np.bool_)).to(device)

print("Initial tensors:")
print("  n:", tuple(initial_n.shape), f"[{initial_n.min().item():.3f}, {initial_n.max().item():.3f}]")
print("  m:", tuple(initial_m.shape), f"[{initial_m.min().item():.3f}, {initial_m.max().item():.3f}]")
print("  s:", tuple(initial_s.shape), f"[{initial_s.min().item():.3f}, {initial_s.max().item():.3f}]")
print("  vessel voxels:", int(vessel_mask.sum().item()))

In [ ]:
# Initialize HemoInvasion3D model with extra-conservative defaults
model = HemoInvasion3D(
    B=torch.tensor(0.010, dtype=torch.float32, device=device),
    Dn=torch.tensor(0.001, dtype=torch.float32, device=device),
    Ds=torch.tensor(0.015, dtype=torch.float32, device=device),
    k_s=torch.tensor(0.040, dtype=torch.float32, device=device),
    s_star=torch.tensor(0.250, dtype=torch.float32, device=device),
    patient_data=patient_data,
    initial_n=initial_n,
    initial_m=initial_m,
    initial_s=initial_s,
    K=torch.tensor(1.0, dtype=torch.float32, device=device),
    s_crit=torch.tensor(0.35, dtype=torch.float32, device=device),
    s_smooth=torch.tensor(0.08, dtype=torch.float32, device=device),
    s_outside=0.0,
    s_vessel=1.0,
    vessel_mask=vessel_mask,
    time_scale_days=120.0,
    poisson_iterations=32,
    require_grad=False,
    device=device,
)

solver = TorchDiffEqSolver(
    model,
    TorchDiffEqSolverOptions(
        step_size=timedelta(days=0.02),
        method="rk4",
        device=device,
        use_adjoint=False,
    ),
)

u0 = model.get_initial_state()
print("u0 shape:", tuple(u0.shape))

In [ ]:
# Mini run: 50 days (daily outputs)
mini_t0 = patient_data.visits[0].time
mini_timepoints = [mini_t0 + timedelta(days=d) for d in range(0, 51)]  # 0..50 days

times_mini, traj_mini = solver.solve(timepoints=mini_timepoints, u_initial=u0)

# Unpack fields: traj shape = (T, 3, D, H, W)
n_series = traj_mini[:, 0]
m_series = traj_mini[:, 1]
s_series = traj_mini[:, 2]

# Physical projection for diagnostics/plots (state constraints)
n_series_phys = torch.clamp(n_series, 0.0, 1.0)
m_series_phys = torch.clamp(m_series, 0.0, 1.0)
s_series_phys = torch.clamp(s_series, 0.0, 1.0)

print("Mini run complete")
print("  trajectory shape:", tuple(traj_mini.shape))
print("  n finite:", bool(torch.isfinite(n_series).all()))
print("  m finite:", bool(torch.isfinite(m_series).all()))
print("  s finite:", bool(torch.isfinite(s_series).all()))

In [ ]:
# Fast sanity checks on dynamics
mass_n = n_series_phys.sum(dim=(1, 2, 3)).detach().cpu().numpy()
mass_m = m_series_phys.sum(dim=(1, 2, 3)).detach().cpu().numpy()
mean_s = s_series_phys.mean(dim=(1, 2, 3)).detach().cpu().numpy()
time_days = times_mini.detach().cpu().numpy()

fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
axes[0].plot(time_days, mass_n)
axes[0].set_title("Total proliferating cells (n)")
axes[0].set_xlabel("days")
axes[0].grid(True, alpha=0.3)

axes[1].plot(time_days, mass_m)
axes[1].set_title("Total quiescent cells (m)")
axes[1].set_xlabel("days")
axes[1].grid(True, alpha=0.3)

axes[2].plot(time_days, mean_s)
axes[2].set_title("Mean substrate (S)")
axes[2].set_xlabel("days")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()

raw_n_min, raw_n_max = n_series.min().item(), n_series.max().item()
raw_m_min, raw_m_max = m_series.min().item(), m_series.max().item()
raw_s_min, raw_s_max = s_series.min().item(), s_series.max().item()

print("Raw ranges (before projection):")
print(f"  n: [{raw_n_min:.4f}, {raw_n_max:.4f}]")
print(f"  m: [{raw_m_min:.4f}, {raw_m_max:.4f}]")
print(f"  S: [{raw_s_min:.4f}, {raw_s_max:.4f}]")
print("Projected ranges (physical):")
print(f"  n: [{n_series_phys.min().item():.4f}, {n_series_phys.max().item():.4f}]")
print(f"  m: [{m_series_phys.min().item():.4f}, {m_series_phys.max().item():.4f}]")
print(f"  S: [{s_series_phys.min().item():.4f}, {s_series_phys.max().item():.4f}]")

# Stability warning for the raw trajectory
max_abs_raw = max(abs(raw_n_min), abs(raw_n_max), abs(raw_m_min), abs(raw_m_max))
if max_abs_raw > 5.0:
    print("WARNING: raw trajectory appears unstable (|n| or |m| > 5).")
    print("Try smaller solver step_size and/or more conservative parameters.")

In [ ]:
# Visual check: center slice for n, m, S at day 0 and day 50
z = n_series.shape[1] // 2

fig, axes = plt.subplots(2, 3, figsize=(12, 7))

def _show(ax, arr, title, cmap):
    im = ax.imshow(arr, cmap=cmap)
    ax.set_title(title)
    ax.axis("off")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

_show(axes[0, 0], n_series_phys[0, z].detach().cpu().numpy(), "n (day 0)", "magma")
_show(axes[0, 1], m_series_phys[0, z].detach().cpu().numpy(), "m (day 0)", "viridis")
_show(axes[0, 2], s_series_phys[0, z].detach().cpu().numpy(), "S (day 0)", "plasma")

_show(axes[1, 0], n_series_phys[-1, z].detach().cpu().numpy(), "n (day 50)", "magma")
_show(axes[1, 1], m_series_phys[-1, z].detach().cpu().numpy(), "m (day 50)", "viridis")
_show(axes[1, 2], s_series_phys[-1, z].detach().cpu().numpy(), "S (day 50)", "plasma")

plt.tight_layout()

## Notes

- This mini run is intended for **fast model sanity checks** only.
- For calibration/forecasting, tune parameters (`B`, `Dn`, `Ds`, `k_s`, transition parameters) and use patient-specific vascular masks if available.
- You can increase horizon and reduce `step_size` after the short run looks stable.

## Full run to last visit

This section mirrors `HGG_Demo` / `TNBC_Demo`: integrate from first to last visit with denser output.  
Then we compare the first 50 days of this full run against the mini-run.

In [ ]:
# Full run: first visit -> last visit (0.5-day output)
full_t0 = patient_data.visits[0].time
full_t1 = patient_data.visits[-1].time

full_timepoints = []
cur_t = full_t0
while cur_t <= full_t1:
    full_timepoints.append(cur_t)
    cur_t += timedelta(days=0.5)
if full_timepoints[-1] != full_t1:
    full_timepoints.append(full_t1)

times_full, traj_full = solver.solve(timepoints=full_timepoints, u_initial=u0)

n_full = traj_full[:, 0]
m_full = traj_full[:, 1]
s_full = traj_full[:, 2]

print("Full run complete")
print("  visits window (days):", (full_t1 - full_t0).days)
print("  output points:", len(full_timepoints))
print("  trajectory shape:", tuple(traj_full.shape))
print("  all finite:", bool(torch.isfinite(traj_full).all()))

In [ ]:
# Compare mini-run vs first 50 days of full-run
full_days = times_full.detach().cpu().numpy()
mini_days = times_mini.detach().cpu().numpy()

mask_50 = full_days <= 50.0
n_mass_full_50 = n_full[mask_50].sum(dim=(1, 2, 3)).detach().cpu().numpy()
m_mass_full_50 = m_full[mask_50].sum(dim=(1, 2, 3)).detach().cpu().numpy()
s_mean_full_50 = s_full[mask_50].mean(dim=(1, 2, 3)).detach().cpu().numpy()

n_mass_mini = n_series.sum(dim=(1, 2, 3)).detach().cpu().numpy()
m_mass_mini = m_series.sum(dim=(1, 2, 3)).detach().cpu().numpy()
s_mean_mini = s_series.mean(dim=(1, 2, 3)).detach().cpu().numpy()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(full_days[mask_50], n_mass_full_50, label="full run (<=50d)", alpha=0.9)
axes[0].plot(mini_days, n_mass_mini, "--", label="mini run 50d", alpha=0.9)
axes[0].set_title("Total n")
axes[0].set_xlabel("days")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].plot(full_days[mask_50], m_mass_full_50, label="full run (<=50d)", alpha=0.9)
axes[1].plot(mini_days, m_mass_mini, "--", label="mini run 50d", alpha=0.9)
axes[1].set_title("Total m")
axes[1].set_xlabel("days")
axes[1].grid(True, alpha=0.3)
axes[1].legend()

axes[2].plot(full_days[mask_50], s_mean_full_50, label="full run (<=50d)", alpha=0.9)
axes[2].plot(mini_days, s_mean_mini, "--", label="mini run 50d", alpha=0.9)
axes[2].set_title("Mean S")
axes[2].set_xlabel("days")
axes[2].grid(True, alpha=0.3)
axes[2].legend()

plt.tight_layout()

# Visual compare center slice at ~day 50
idx50_full = int(np.argmin(np.abs(full_days - 50.0)))
idx50_mini = int(np.argmin(np.abs(mini_days - 50.0)))
z = n_series.shape[1] // 2

fig, axes = plt.subplots(2, 3, figsize=(12, 7))

def _imshow(ax, arr, title, cmap):
    im = ax.imshow(arr, cmap=cmap)
    ax.set_title(title)
    ax.axis("off")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

_imshow(axes[0, 0], n_full[idx50_full, z].detach().cpu().numpy(), "n full ~day50", "magma")
_imshow(axes[0, 1], m_full[idx50_full, z].detach().cpu().numpy(), "m full ~day50", "viridis")
_imshow(axes[0, 2], s_full[idx50_full, z].detach().cpu().numpy(), "S full ~day50", "plasma")

_imshow(axes[1, 0], n_series[idx50_mini, z].detach().cpu().numpy(), "n mini day50", "magma")
_imshow(axes[1, 1], m_series[idx50_mini, z].detach().cpu().numpy(), "m mini day50", "viridis")
_imshow(axes[1, 2], s_series[idx50_mini, z].detach().cpu().numpy(), "S mini day50", "plasma")

plt.tight_layout()

## Parameter calibration (LM) on early visits

Below is a lightweight calibration workflow inspired by `HGG_Demo` and `TNBC_Demo`:

- use early visits as targets,
- optimize a small set of PDE parameters,
- track loss and compare pre/post calibration trajectories.

To keep runtime practical, the default example calibrates only 3 parameters (`B`, `Dn`, `k_s`) on the first 3 visits.

In [ ]:
# Build measured target maps at visit times
measured_cellularity_maps = [
    ADC_to_cellularity(v.adc_image, v.roi_enhance_image, v.roi_nonenhance_image)
    for v in patient_data.visits
]

n_visits_calibration = min(3, len(patient_data.visits))  # include initial visit
target_timepoints = [v.time for v in patient_data.visits[:n_visits_calibration]]

y_target = torch.stack(
    [
        torch.from_numpy(measured_cellularity_maps[i].array).float().to(device)
        for i in range(n_visits_calibration)
    ],
    dim=0,
)

print("Calibration targets prepared")
print("  visits used:", n_visits_calibration)
print("  target shape:", tuple(y_target.shape))

In [ ]:
# Helper: update model params -> predict n(t, x)
def update_model_and_predict(model_parameters, timepoints=target_timepoints):
    # model_parameters = [B, Dn, k_s]
    B_val, Dn_val, ks_val = model_parameters

    model.B.data = torch.tensor(float(B_val), dtype=torch.float32, device=device)
    model.Dn.data = torch.tensor(float(Dn_val), dtype=torch.float32, device=device)
    model.k_s.data = torch.tensor(float(ks_val), dtype=torch.float32, device=device)

    _, traj = solver.solve(timepoints=timepoints, u_initial=model.get_initial_state())
    pred_n = torch.clamp(traj[:, 0], 0.0, 1.0)
    return pred_n


# Optional baseline error before calibration
params_pre_cal = torch.tensor([model.B.item(), model.Dn.item(), model.k_s.item()], dtype=torch.float64)
pred0 = update_model_and_predict(params_pre_cal)
baseline_sse = torch.sum((pred0 - y_target) ** 2).item()
print(f"Baseline SSE: {baseline_sse:.4e}")

In [ ]:
# LM calibration setup
initial_parameters = torch.tensor(
    [model.B.item(), model.Dn.item(), model.k_s.item()], dtype=torch.float64
)

# bounds for [B, Dn, k_s]
bounds = torch.tensor(
    [
        [0.005, 0.080],   # B
        [0.0005, 0.020],  # Dn
        [0.020, 0.300],   # k_s
    ],
    dtype=torch.float64,
)

lm_options = LMoptions(
    jac_delta=1e-4,
    jac_update_interval=1,
    lambda_init=1.0,
    lambda_upscale_factor=5.0,
    lambda_downscale_factor=1.5,
)

optim = LMoptimizer(
    model=update_model_and_predict,
    bounds=bounds,
    initial_guess=initial_parameters,
    y_data=y_target,
    options=lm_options,
)

n_optim_steps = 6
for i in range(n_optim_steps):
    optim.step()
    print(f"step {i+1:02d}: best_error={optim.error[-1]:.4e}, params={optim.parameters[-1].tolist()}")

best_parameters = optim.parameters[-1]
print("Best parameters:", best_parameters)

In [ ]:
# Apply best parameters and compare against targets
model.B.data = torch.tensor(float(best_parameters[0]), dtype=torch.float32, device=device)
model.Dn.data = torch.tensor(float(best_parameters[1]), dtype=torch.float32, device=device)
model.k_s.data = torch.tensor(float(best_parameters[2]), dtype=torch.float32, device=device)

pred_cal = update_model_and_predict(best_parameters)
cal_sse = torch.sum((pred_cal - y_target) ** 2).item()

print(f"Baseline SSE:   {baseline_sse:.4e}")
print(f"Calibrated SSE: {cal_sse:.4e}")

# Loss trace
plt.figure(figsize=(6, 3.5))
plt.plot(optim.error, marker="o")
plt.title("LM calibration loss (best SSE)")
plt.xlabel("iteration")
plt.grid(True, alpha=0.3)
plt.tight_layout()

# Compare mean tumor density at calibration visits
pred_mean = pred_cal.mean(dim=(1, 2, 3)).detach().cpu().numpy()
tgt_mean = y_target.mean(dim=(1, 2, 3)).detach().cpu().numpy()
visit_days = np.array([(t - target_timepoints[0]).days for t in target_timepoints], dtype=float)

plt.figure(figsize=(6, 3.5))
plt.plot(visit_days, tgt_mean, "o-", label="target mean n")
plt.plot(visit_days, pred_mean, "s--", label="predicted mean n (calibrated)")
plt.title("Calibration fit at visit times")
plt.xlabel("days from first visit")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()

## Quick vs full calibration presets

This section adds two ready-to-run calibration modes:

- `quick`: fewer LM iterations and fewer visits (fast debug)
- `full`: more iterations and more visits (better fit, slower)

Run one preset and then compare **day-50 tumor slice before vs after calibration**.

In [ ]:
calibration_presets = {
    "quick": {
        "n_visits": min(3, len(patient_data.visits)),
        "n_steps": 4,
        "solver_step_days": 0.06,
    },
    "full": {
        "n_visits": min(5, len(patient_data.visits)),
        "n_steps": 12,
        "solver_step_days": 0.03,
    },
}


def run_hemo_calibration(preset_name="quick"):
    cfg = calibration_presets[preset_name]

    # Save current params to compare before/after
    params_before = torch.tensor(
        [model.B.item(), model.Dn.item(), model.k_s.item()], dtype=torch.float64
    )

    target_t = [v.time for v in patient_data.visits[: cfg["n_visits"]]]
    y_tgt = torch.stack(
        [
            torch.from_numpy(measured_cellularity_maps[i].array).float().to(device)
            for i in range(cfg["n_visits"])
        ],
        dim=0,
    )

    # Temporarily adjust solver step for calibration workload
    old_step = solver.solver_options.step_size
    solver.solver_options.step_size = timedelta(days=cfg["solver_step_days"])

    def _predict(params, timepoints=target_t):
        B_val, Dn_val, ks_val = params
        model.B.data = torch.tensor(float(B_val), dtype=torch.float32, device=device)
        model.Dn.data = torch.tensor(float(Dn_val), dtype=torch.float32, device=device)
        model.k_s.data = torch.tensor(float(ks_val), dtype=torch.float32, device=device)
        _, traj = solver.solve(timepoints=timepoints, u_initial=model.get_initial_state())
        return torch.clamp(traj[:, 0], 0.0, 1.0)

    init_guess = params_before.clone()
    bounds = torch.tensor(
        [[0.003, 0.060], [0.0003, 0.010], [0.015, 0.200]], dtype=torch.float64
    )

    optim_local = LMoptimizer(
        model=_predict,
        bounds=bounds,
        initial_guess=init_guess,
        y_data=y_tgt,
        options=LMoptions(jac_delta=1e-4, jac_update_interval=1, lambda_init=1.0),
    )

    for _ in range(cfg["n_steps"]):
        optim_local.step()

    best = optim_local.parameters[-1]
    model.B.data = torch.tensor(float(best[0]), dtype=torch.float32, device=device)
    model.Dn.data = torch.tensor(float(best[1]), dtype=torch.float32, device=device)
    model.k_s.data = torch.tensor(float(best[2]), dtype=torch.float32, device=device)

    baseline = torch.sum((_predict(params_before) - y_tgt) ** 2).item()
    calibrated = torch.sum((_predict(best) - y_tgt) ** 2).item()

    # restore solver step
    solver.solver_options.step_size = old_step

    return {
        "preset": preset_name,
        "config": cfg,
        "params_before": params_before,
        "params_after": best,
        "baseline_sse": baseline,
        "calibrated_sse": calibrated,
        "optim": optim_local,
    }


# Choose preset: "quick" or "full"
cal_result = run_hemo_calibration("quick")
print("Preset:", cal_result["preset"])
print("Params before:", cal_result["params_before"])
print("Params after :", cal_result["params_after"])
print(f"SSE before: {cal_result['baseline_sse']:.4e}")
print(f"SSE after : {cal_result['calibrated_sse']:.4e}")

plt.figure(figsize=(6, 3.2))
plt.plot(cal_result["optim"].error, marker="o")
plt.title(f"LM loss ({cal_result['preset']})")
plt.xlabel("iteration")
plt.grid(True, alpha=0.3)
plt.tight_layout()

In [ ]:
# Day-50 tumor slice before vs after calibration

def run_50d_n_series(params, step_days=0.05):
    old_step = solver.solver_options.step_size
    solver.solver_options.step_size = timedelta(days=step_days)

    model.B.data = torch.tensor(float(params[0]), dtype=torch.float32, device=device)
    model.Dn.data = torch.tensor(float(params[1]), dtype=torch.float32, device=device)
    model.k_s.data = torch.tensor(float(params[2]), dtype=torch.float32, device=device)

    t0 = patient_data.visits[0].time
    tp = [t0 + timedelta(days=d) for d in range(0, 51)]
    tdays, traj = solver.solve(timepoints=tp, u_initial=model.get_initial_state())

    solver.solver_options.step_size = old_step
    return tdays.detach().cpu().numpy(), torch.clamp(traj[:, 0], 0.0, 1.0)


_, n_before = run_50d_n_series(cal_result["params_before"], step_days=0.05)
_, n_after = run_50d_n_series(cal_result["params_after"], step_days=0.05)

z = n_before.shape[1] // 2
idx50 = 50

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

im0 = axes[0].imshow(n_before[idx50, z].detach().cpu().numpy(), cmap="magma")
axes[0].set_title("n day50 before calibration")
axes[0].axis("off")
plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

im1 = axes[1].imshow(n_after[idx50, z].detach().cpu().numpy(), cmap="magma")
axes[1].set_title("n day50 after calibration")
axes[1].axis("off")
plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

delta = (n_after[idx50, z] - n_before[idx50, z]).detach().cpu().numpy()
im2 = axes[2].imshow(delta, cmap="coolwarm")
axes[2].set_title("delta (after - before)")
axes[2].axis("off")
plt.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)

plt.tight_layout()